# Занятие 3, пара 1. Свёртка и линейные фильтры


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


Помните: картинка для компьютера — массив чисел. Сегодня немного поработаем с соседями каждого пикселя. Иногда соседи помогают убрать шум, иногда дружно размывают всё полезное. Давайте проверим, от чего это зависит.

Нужны NumPy, OpenCV и Matplotlib. Интернет, GPU и фото с телефона не нужны. В конце вы сможете объяснить действие небольшого ядра и выбрать параметры простого фильтра.


### Подготовка

Работайте в своей копии внутри папки проекта. Запустите ячейки по порядку. Эта тетрадь не использует переменные из другой тетради. GitHub — по желанию; сохраняйте файл и отдельную копию.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from course_support.cvtools import imread, imwrite, show
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'filters'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
image = imread(DATA_DIR / 'hopper.jpg', cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (256, 300), interpolation=cv2.INTER_AREA)
print(image.shape, image.dtype, image.min(), image.max())
show(image, titles=['Исходное изображение'])


## Девять соседей
Возьмём маленький участок: в центре яркий пиксель 90, вокруг восемь пикселей по 10. Усреднение складывает девять значений и делит на девять. Сначала прикиньте ответ без запуска.

Ту же операцию можно записать как умножение каждого пикселя на вес и сумму произведений. Таблица весов называется **ядром**. Здесь все веса равны 1/9.


In [ ]:
patch = np.array([[10, 10, 10], [10, 90, 10], [10, 10, 10]], dtype=np.float64)
box_kernel = np.ones((3, 3), dtype=np.float64) / 9
print('Пиксели:'); print(patch)
print('Веса:'); print(box_kernel)
print('Сумма весов:', box_kernel.sum())


### ✏️ Ваш ход 1: один новый пиксель

Вычислите `center_value = сумма(patch * box_kernel)`. Затем создайте ядро `identity_kernel` 3×3, которое оставит центральный пиксель без изменения: какой вес поставить в центре, а какие вокруг?
После проверки измените 90 на 180 в копии patch и предскажите результат усреднения.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.1', center_value, identity_kernel)


### Можно остановиться здесь
Сохраните тетрадь и свой ответ. Уже понятно, как одно ядро превращает девять чисел в один новый пиксель. Дальше пройдём тем же окошком по изображению.


## Окошко едет по изображению
Функция ниже уже написана. Давайте проследим одну итерацию: вырезать окошко → умножить на веса → сложить → положить результат в новый массив.

Работаем с float, чтобы не потерять дроби и отрицательные значения. Для края используем отражение без повторения крайнего пикселя: `np.pad(..., mode='reflect')` соответствует `cv2.BORDER_REFLECT_101`.

Небольшая точность в названии: `filter2D` применяет **корреляцию**, то есть не переворачивает ядро. Для математической свёртки ядро переворачивают по двум осям. У симметричного ядра размытия результаты совпадают. Дальше сравниваем одинаковые операции.


In [ ]:
def manual_filter(image, kernel):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    height, width = image.shape
    kh, kw = kernel.shape
    if kh % 2 == 0 or kw % 2 == 0:
        raise ValueError('В этом примере нужны нечётные размеры ядра')
    padded = np.pad(image, ((kh//2, kh//2), (kw//2, kw//2)), mode='reflect')
    result = np.zeros_like(image)
    for y in range(height):
        for x in range(width):
            region = padded[y:y+kh, x:x+kw]
            result[y, x] = np.sum(region * kernel)
    return result

small = np.arange(25, dtype=np.float64).reshape(5, 5)
print('Исходный массив:'); print(small)
print('После усреднения:'); print(np.round(manual_filter(small, box_kernel), 2))


### ✏️ Ваш ход 2: доверяем, но сверяем

Получите `manual_result` через manual_filter и `opencv_result` через `cv2.filter2D(small, cv2.CV_64F, box_kernel, borderType=cv2.BORDER_REFLECT_101)`. Вычислите максимальную абсолютную разницу. Объясните, зачем одинаково задать обработку края.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.2', manual_result, opencv_result)


## Что находится за краем фотографии?
За краем массива данных нет. Фильтр всё равно должен что-то получить для своего окошка. Сравним чёрное продолжение и отражение на постоянной картинке. Какой способ создаст тёмную рамку?


In [ ]:
constant = np.full((7, 7), 100.0)
black_border = cv2.filter2D(constant, -1, box_kernel, borderType=cv2.BORDER_CONSTANT)
reflected = cv2.filter2D(constant, -1, box_kernel, borderType=cv2.BORDER_REFLECT_101)
print('Верхний левый пиксель:', black_border[0, 0], reflected[0, 0])
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, result, title in zip(axes, [black_border, reflected], ['Нули снаружи', 'Отражение']):
    ax.imshow(result, cmap='gray', vmin=0, vmax=100); ax.set_title(title); ax.axis('off')
plt.tight_layout()


## Размыть или сделать резче?
Сохраним знакомые ядра из старого примера, но посмотрим на три эффекта по очереди. Усреднение 3×3 и 9×9 смешивает соседей с разной дальностью. Ядро резкости усиливает центр и вычитает соседей.

Почему у этих ядер сумма равна 1? На постоянном участке яркость должна остаться прежней. Но это не значит, что каждое новое значение останется в диапазоне 0…255. Для показа обрежем значения, а для вычислений сохраним float.


In [ ]:
image_float = image.astype(np.float64)
kernels = {
    'Среднее 3×3': np.ones((3, 3)) / 9,
    'Среднее 9×9': np.ones((9, 9)) / 81,
    'Резкость': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
}
filtered = {name: cv2.filter2D(image_float, -1, kernel, borderType=cv2.BORDER_REFLECT_101)
            for name, kernel in kernels.items()}
show(image, *[np.clip(value, 0, 255).astype(np.uint8) for value in filtered.values()],
     titles=['Исходное', *filtered.keys()])
print('Диапазон до обрезки:', filtered['Резкость'].min(), filtered['Резкость'].max())


### ✏️ Ваш ход 3: регулируем резкость

Создайте ядро `sharp_kernel` для `a=0.5`: четыре соседа равны `-a`, углы 0, центр `1 + 4*a`. Примените к `image_float`, сохраните float-результат в `sharp_result`, покажите после clip.
Затем по очереди попробуйте a=0 и a=1. Напишите: какой вариант выбрали и где усиление стало мешать. Проверку выполняйте для a=0.5, а опыты — отдельной ячейкой.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.3', sharp_kernel, sharp_result, image_float)


## Гауссово размытие: ближние соседи важнее
При обычном среднем все пиксели окна равноправны. Гауссово ядро даёт ближним больший вес. `sigma` задаёт ширину распределения весов; размер окна тоже ограничивает, насколько далеко мы смотрим.

Сначала сравним три значения sigma при одном окне 9×9. Где исчезает нужная деталь? Угадываем до запуска, потом проверяем.


In [ ]:
sigmas = [0.6, 1.5, 3.0]
gaussian = [cv2.GaussianBlur(image, (9, 9), sigmaX=s, borderType=cv2.BORDER_REFLECT_101) for s in sigmas]
show(image, *gaussian, titles=['Исходное', *[f'sigma={s}' for s in sigmas]])
imwrite(OUTPUT_DIR / 'sharpened.png', np.clip(sharp_result, 0, 255).astype(np.uint8))


## Практический запас — 15–20 минут
**Если интересно, откуда берутся веса.** Эта функция из прежнего примера строит ядро по формуле. Запустите для size=7, sigma=1.5 и сравните с внешним произведением одномерного ядра OpenCV. Основная часть пары от этого опыта не зависит.


In [ ]:
def create_gaussian_kernel(size, sigma):
    kernel = np.zeros((size, size))
    center = size // 2
    for i in range(size):
        for j in range(size):
            x, y = i - center, j - center
            kernel[i, j] = np.exp(-(x*x + y*y) / (2 * sigma*sigma))
    return kernel / kernel.sum()

custom = create_gaussian_kernel(7, 1.5)
one_dim = cv2.getGaussianKernel(7, 1.5)
print('Сумма:', custom.sum(), 'максимальная разница:', np.abs(custom - one_dim @ one_dim.T).max())
small_crop = image_float[:32, :32]
direct = cv2.filter2D(small_crop, -1, custom, borderType=cv2.BORDER_REFLECT_101)
separate = cv2.sepFilter2D(small_crop, -1, one_dim, one_dim, borderType=cv2.BORDER_REFLECT_101)
print('Два прохода и полное ядро:', np.max(np.abs(direct - separate)))


## Итог
Сохраните тетрадь и одну картинку. В Markdown: что делают веса, зачем мы указали край и какой эффект выбрали для изображения. Никаких баллов за скорость. Это подготовка к ЛР2; края и контуры разберём в следующем блоке.

В нейросети веса похожих локальных операций будут обучаться по данным. До этого ещё дойдём; сегодня важно видеть действие конкретных чисел.

По желанию: [OpenCV — сглаживание](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html), Ричард Сзелиски, [Computer Vision: Algorithms and Applications](https://szeliski.org/Book/) — раздел об обработке изображений; не нужно читать всю книгу к следующей паре.


In [ ]:
summary('03.1')
